# Phase 2: Embed All Models into FLUX

**Objective:** Embed all language, vision, and audio models into a single self-contained .flx file.

**Models to Embed:**
1. **Qwen2.5-1.5B-Instruct** (~3 GB) - Main reasoning voice
2. **Qwen2-VL-2B** (~4 GB) - Vision + language (smaller than 3B)\n
3. **Qwen2.5-Coder-1.5B** (~3 GB) - Code generation
4. **Whisper-small** (~500 MB) - Speech to text
5. **Bark-small** (~500 MB) - Text to speech
6. **all-MiniLM-L6-v2** (~100 MB) - Wave embedding
7. **CLIP ViT-L/14** (~900 MB) - Vision-language bridge

**Input:** `Flux-Lite-Base.flx` (~500 MB compressed native)
**Output:** `Flux-Apex-V1.flx` (~14-17 GB all-in-one)

In [ ]:
"""Cell 1: Environment Setup"""

import os
import sys
import gc
import torch
from pathlib import Path
from datetime import datetime
from typing import Dict, Any, Optional, Type

# Environment detection
if Path('/kaggle').exists():
    ROOT = Path('/kaggle/working/FLUX')
    ENVIRONMENT = 'kaggle'
    if not ROOT.exists():
        os.system('git clone https://github.com/Unseengap/FLUX.git /kaggle/working/FLUX')
elif Path('/content').exists():
    ROOT = Path('/content/FLUX')
    ENVIRONMENT = 'colab'
    if not ROOT.exists():
        os.system('git clone https://github.com/Unseengap/FLUX.git /content/FLUX')
else:
    ROOT = Path('/Users/admin/Desktop/flux')
    ENVIRONMENT = 'local'

sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

# Install dependencies if needed
try:
    from transformers import AutoModelForCausalLM
except ImportError:
    os.system('pip install -q transformers accelerate sentencepiece')

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    os.system('pip install -q sentence-transformers')

print(f"Environment: {ENVIRONMENT}")
print(f"Root: {ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
"""Cell 2: Initialize Logger & Token"""

from flux_utils import PhaseLogger, get_device, _resolve_hf_token

log = PhaseLogger(phase=201)  # Phase 201 = Model embedding
log.separator("FLUX Model Embedding - Phase 2")

device = get_device()
log.info(f"Device: {device}")

# HuggingFace token
try:
    hf_token = _resolve_hf_token()
    log.success("HuggingFace token resolved")
except:
    hf_token = None
    log.warning("No HF token - some models may not download")

In [ ]:
"""Cell 3: Load FLUX Lite Base"""

log.cell_start("Cell 3 — Load Base")

from huggingface_hub import hf_hub_download

# Try lite base first, fall back to full model
LITE_PATH = ROOT / 'checkpoints' / 'Flux-Lite-Base.flx'
FULL_PATH = ROOT / 'checkpoints' / 'Flux-Apex-V1.flx'

if LITE_PATH.exists():
    MODEL_PATH = LITE_PATH
    log.info(f"Using Lite base: {LITE_PATH}")
elif FULL_PATH.exists():
    MODEL_PATH = FULL_PATH
    log.warning(f"Lite base not found, using full model: {FULL_PATH}")
else:
    log.info("Downloading from HuggingFace...")
    FULL_PATH.parent.mkdir(parents=True, exist_ok=True)
    downloaded = hf_hub_download(
        repo_id='UnseenGAP/FLUX',
        filename='checkpoints/Flux-Apex-V1.flx',
        local_dir=str(ROOT),
        token=hf_token,
    )
    MODEL_PATH = Path(downloaded)
    log.success(f"Downloaded: {MODEL_PATH}")

# Load
flux_model = torch.load(str(MODEL_PATH), map_location='cpu', weights_only=False)
log.info(f"Version: {flux_model.get('version', 'unknown')}")
log.info(f"Size: {MODEL_PATH.stat().st_size / 1e9:.2f} GB")

# Initialize models section
if 'models' not in flux_model:
    flux_model['models'] = {}

log.cell_end("Cell 3 — Load Base", "PASS")

In [ ]:
"""Cell 4: Define Model Embedding Utilities"""

log.cell_start("Cell 4 — Embedding Utilities")

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    AutoProcessor,
    AutoConfig,
)

def embed_hf_model(
    model_id: str,
    model_class: Type = AutoModelForCausalLM,
    processor_class: Optional[Type] = None,
    quantization: str = 'fp16',
    device: str = 'cuda',
    trust_remote_code: bool = True,
) -> Dict[str, Any]:
    """
    Download HuggingFace model and extract weights for .flx embedding.
    
    Args:
        model_id: HuggingFace model identifier
        model_class: Model class to use for loading
        processor_class: Optional processor/tokenizer class
        quantization: 'fp16' or 'fp32'
        device: Device for initial loading
        trust_remote_code: Trust custom model code
    
    Returns:
        Dict ready for .flx storage with weights, config, metadata
    """
    print(f"\n  Loading {model_id}...")
    
    dtype = torch.float16 if quantization == 'fp16' else torch.float32
    
    # Load model
    try:
        hf_model = model_class.from_pretrained(
            model_id,
            torch_dtype=dtype,
            trust_remote_code=trust_remote_code,
            low_cpu_mem_usage=True,
            token=hf_token,
        )
    except Exception as e:
        print(f"    ⚠ Primary load failed: {e}")
        print(f"    Trying with device_map='auto'...")
        hf_model = model_class.from_pretrained(
            model_id,
            torch_dtype=dtype,
            device_map='auto',
            trust_remote_code=trust_remote_code,
            low_cpu_mem_usage=True,
            token=hf_token,
        )
    
    # Extract weights to CPU
    weights = {}
    total_params = 0
    for name, param in hf_model.named_parameters():
        weights[name] = param.detach().cpu()
        total_params += param.numel()
    
    # Get config
    config = {}
    if hasattr(hf_model, 'config'):
        try:
            config = hf_model.config.to_dict()
        except:
            config = {'model_type': getattr(hf_model.config, 'model_type', 'unknown')}
    
    size_gb = total_params * (2 if quantization == 'fp16' else 4) / 1e9
    print(f"    ✓ Loaded: {total_params:,} params ({size_gb:.2f} GB {quantization})")
    
    # Cleanup GPU memory
    del hf_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return {
        'base_model': model_id,
        'quantization': quantization,
        'total_params': total_params,
        'weights': weights,
        'config': config,
        'lazy_load': True,
    }


def get_component_size(component: Any) -> int:
    """Calculate size of a component in bytes."""
    total = 0
    if isinstance(component, torch.Tensor):
        return component.numel() * component.element_size()
    elif isinstance(component, dict):
        for v in component.values():
            total += get_component_size(v)
    elif isinstance(component, (list, tuple)):
        for item in component:
            total += get_component_size(item)
    return total


print("  ✓ Embedding utilities ready")
log.cell_end("Cell 4 — Embedding Utilities", "PASS")

In [ ]:
"""Cell 5: Embed Instruct Model (Main Voice)"""

log.cell_start("Cell 5 — Embed Instruct")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

instruct_state = embed_hf_model(
    model_id='Qwen/Qwen2.5-1.5B-Instruct',
    model_class=AutoModelForCausalLM,
    quantization='fp16',
)

flux_model['models']['instruct'] = {
    **instruct_state,
    'role': 'main_voice',
    'tasks': ['tool_calling', 'reasoning', 'instruction_following', 'conversation'],
    'lazy_load': False,  # Always loaded - main voice
}

log.success(f"Instruct embedded: {instruct_state['total_params']:,} params")
log.cell_end("Cell 5 — Embed Instruct", "PASS")

In [ ]:
"""Cell 6: Replace VLM with Fresh 2B Model"""

log.cell_start("Cell 6 - VLM")

# ALWAYS embed fresh 2B VLM (replacing any existing 3B)
# Delete old vlm section if present (it's the 3B from before)
if 'vlm' in flux_model:
    old_model = flux_model['vlm'].get('base_model', 'unknown')
    print(f"  Removing old VLM: {old_model}")
    del flux_model['vlm']
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Embed fresh Qwen2-VL-2B (NOTE: Qwen2, not Qwen2.5 - 2.5 has no 2B variant)
print("  Embedding fresh Qwen2-VL-2B-Instruct...")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Use Qwen2VLForConditionalGeneration (NOT Qwen2_5_VLForConditionalGeneration)
from transformers import Qwen2VLForConditionalGeneration

vlm_state = embed_hf_model(
    model_id='Qwen/Qwen2-VL-2B-Instruct',
    model_class=Qwen2VLForConditionalGeneration,
    quantization='fp16',
)

flux_model['models']['vision'] = {
    **vlm_state,
    'role': 'vision_language',
    'tasks': ['image_understanding', 'visual_qa', 'grid_analysis'],
    'lazy_load': True,
}

log.success(f"VLM embedded (Qwen2-VL-2B): {vlm_state['total_params']:,} params")
log.cell_end("Cell 6 - VLM", "PASS")

In [ ]:
"""Cell 7: Embed Coder Model"""

log.cell_start("Cell 7 — Embed Coder")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

coder_state = embed_hf_model(
    model_id='Qwen/Qwen2.5-Coder-1.5B-Instruct',
    model_class=AutoModelForCausalLM,
    quantization='fp16',
)

flux_model['models']['coder'] = {
    **coder_state,
    'role': 'code_generation',
    'tasks': ['write_python', 'generate_transform', 'code_explanation', 'debug'],
    'lazy_load': True,
}

log.success(f"Coder embedded: {coder_state['total_params']:,} params")
log.cell_end("Cell 7 — Embed Coder", "PASS")

In [ ]:
"""Cell 8: Embed Whisper (Speech-to-Text)"""

log.cell_start("Cell 8 — Embed Whisper")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from transformers import WhisperForConditionalGeneration

whisper_state = embed_hf_model(
    model_id='openai/whisper-small',
    model_class=WhisperForConditionalGeneration,
    quantization='fp16',
    trust_remote_code=False,
)

flux_model['models']['whisper'] = {
    **whisper_state,
    'role': 'speech_to_text',
    'tasks': ['transcribe_audio', 'voice_input'],
    'lazy_load': True,
}

log.success(f"Whisper embedded: {whisper_state['total_params']:,} params")
log.cell_end("Cell 8 — Embed Whisper", "PASS")

In [ ]:
"""Cell 9: Embed TTS (Text-to-Speech)"""

log.cell_start("Cell 9 — Embed TTS")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Bark is the TTS model
try:
    from transformers import BarkModel
    
    tts_state = embed_hf_model(
        model_id='suno/bark-small',
        model_class=BarkModel,
        quantization='fp16',
        trust_remote_code=False,
    )
    
    flux_model['models']['tts'] = {
        **tts_state,
        'role': 'text_to_speech',
        'tasks': ['speak', 'voice_output', 'read_aloud'],
        'lazy_load': True,
    }
    
    log.success(f"TTS (Bark) embedded: {tts_state['total_params']:,} params")
    
except Exception as e:
    log.warning(f"TTS embedding failed: {e}")
    log.info("TTS will be loaded from HuggingFace on demand")
    
    flux_model['models']['tts'] = {
        'base_model': 'suno/bark-small',
        'role': 'text_to_speech',
        'tasks': ['speak', 'voice_output'],
        'lazy_load': True,
        'weights': None,  # Load from HF on demand
    }

log.cell_end("Cell 9 — Embed TTS", "PASS")

In [ ]:
"""Cell 10: Embed Sentence Transformer (Wave Encoder)"""

log.cell_start("Cell 10 — Embed Embedding Model")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from sentence_transformers import SentenceTransformer

# Load sentence transformer
st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Extract weights
st_weights = {}
total_params = 0
for name, param in st_model.named_parameters():
    st_weights[name] = param.detach().cpu().half()
    total_params += param.numel()

flux_model['models']['embedding'] = {
    'base_model': 'sentence-transformers/all-MiniLM-L6-v2',
    'quantization': 'fp16',
    'total_params': total_params,
    'weights': st_weights,
    'config': {
        'output_dim': 384,
        'max_seq_length': 256,
    },
    'role': 'wave_encoder',
    'tasks': ['text_to_embedding', 'semantic_similarity', 'wave_conversion'],
    'lazy_load': False,  # Always loaded - used for all wave operations
}

del st_model
gc.collect()

log.success(f"Embedding model embedded: {total_params:,} params")
log.cell_end("Cell 10 — Embed Embedding Model", "PASS")

In [ ]:
"""Cell 11: Embed CLIP (Vision-Language Bridge)"""

log.cell_start("Cell 11 — Embed CLIP")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from transformers import CLIPModel

clip_state = embed_hf_model(
    model_id='openai/clip-vit-large-patch14',
    model_class=CLIPModel,
    quantization='fp16',
    trust_remote_code=False,
)

flux_model['models']['clip'] = {
    **clip_state,
    'role': 'vision_language_bridge',
    'tasks': ['image_embedding', 'text_embedding', 'zero_shot_classification'],
    'lazy_load': False,  # Always loaded - bridge for all vision
}

log.success(f"CLIP embedded: {clip_state['total_params']:,} params")
log.cell_end("Cell 11 — Embed CLIP", "PASS")

In [ ]:
"""Cell 12: Add Lazy Loader Infrastructure"""

log.cell_start("Cell 12 — Lazy Loader")

# Store the lazy loader code in the model for reference
LAZY_LOADER_CODE = '''
class EmbeddedLazyModel:
    """
    Lazy loader for models embedded in .flx files.
    
    Usage:
        loader = EmbeddedLazyModel('coder', flux_model['models']['coder'])
        loader.load()  # Downloads architecture, loads embedded weights
        response = loader.generate("Write hello world")
        loader.unload()  # Free GPU memory
    """
    
    def __init__(self, name: str, model_state: dict):
        self.name = name
        self.config = model_state
        self._model = None
        self._processor = None
        self._loaded = False
    
    @property
    def is_loaded(self) -> bool:
        return self._loaded
    
    def load(self, device: str = 'cuda'):
        """Load model architecture from HF, weights from .flx."""
        if self._loaded:
            return
        
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        
        base_model = self.config['base_model']
        
        # Get processor/tokenizer
        self._processor = AutoTokenizer.from_pretrained(
            base_model, trust_remote_code=True
        )
        
        # Load architecture (cached after first download)
        self._model = AutoModelForCausalLM.from_pretrained(
            base_model,
            torch_dtype=torch.float16,
            device_map='auto' if device == 'cuda' else None,
            trust_remote_code=True,
            low_cpu_mem_usage=True,
        )
        
        # Load embedded weights (no network!)
        if self.config.get('weights'):
            self._model.load_state_dict(self.config['weights'], strict=False)
        
        self._model.eval()
        self._loaded = True
    
    def unload(self):
        """Free GPU memory."""
        if self._model is not None:
            del self._model
            self._model = None
            self._loaded = False
            import gc
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    def generate(self, prompt: str, max_tokens: int = 512, **kwargs):
        """Generate text response."""
        if not self._loaded:
            self.load()
        
        inputs = self._processor(prompt, return_tensors='pt').to(self._model.device)
        outputs = self._model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            **kwargs
        )
        return self._processor.decode(outputs[0], skip_special_tokens=True)
'''

if 'orchestration' not in flux_model:
    flux_model['orchestration'] = {}

flux_model['orchestration']['lazy_loader_code'] = LAZY_LOADER_CODE

print("  ✓ Lazy loader infrastructure added")
log.cell_end("Cell 12 — Lazy Loader", "PASS")

In [ ]:
"""Cell 13: Update Orchestration Config"""

log.cell_start("Cell 13 — Orchestration")

# Model routing triggers
flux_model['orchestration']['model_triggers'] = {
    # Which keywords trigger which model
    'vision': ['image', 'picture', 'see', 'look', 'visual', 'photo', 'what is this', 'describe'],
    'coder': ['code', 'python', 'function', 'implement', 'write', 'script', 'program', 'debug'],
    'whisper': ['audio', 'voice', 'hear', 'listen', 'transcribe', 'speech'],
    'tts': ['speak', 'say', 'read aloud', 'voice output', 'tell me', 'pronounce'],
}

# Load policies
flux_model['orchestration']['always_loaded'] = ['instruct', 'embedding', 'clip']
flux_model['orchestration']['lazy_models'] = ['vision', 'coder', 'whisper', 'tts']

# Architecture type
flux_model['orchestration']['architecture'] = 'multi_model_embedded'

# System prompt for tool calling
flux_model['orchestration']['system_prompt'] = '''You are FLUX, a cognitive AI with multiple specialized capabilities.

You have access to these embedded models:
- instruct: Main reasoning and conversation
- vision: Image understanding and visual QA  
- coder: Python code generation and debugging
- whisper: Speech-to-text transcription
- tts: Text-to-speech output
- embedding: Semantic similarity and wave encoding
- clip: Zero-shot image classification

And these FLUX-native tools:
- encode_grid: Encode ARC grid to wave representation
- query_field: Query the resonance field for knowledge
- recall_memory: Recall from episodic memory
- store_memory: Store to episodic memory
- predict_effect: Use causal graph to predict outcomes

To use a tool, output:
<tool>tool_name</tool>
<params>{"key": "value"}</params>

Think step by step and use tools when they would help answer the query.
'''

print(f"  Models configured: {list(flux_model['models'].keys())}")
print(f"  Always loaded: {flux_model['orchestration']['always_loaded']}")
print(f"  Lazy loaded: {flux_model['orchestration']['lazy_models']}")

log.cell_end("Cell 13 — Orchestration", "PASS")

In [ ]:
"""Cell 14: Update Version & Metadata"""

log.cell_start("Cell 14 — Metadata")

# Update version
flux_model['version'] = '6.0-multi-model-embedded'
flux_model['phase'] = 'phase_multi_model_embed'
flux_model['timestamp'] = datetime.now().isoformat()

# Update metadata
if 'metadata' not in flux_model:
    flux_model['metadata'] = {}

flux_model['metadata']['last_modified'] = datetime.now().isoformat()
flux_model['metadata']['modified_components'] = ['models', 'orchestration']

# Add capabilities
new_caps = [
    'multi_model_embedded',
    'offline_capable',
    'instruct', 'vision', 'coder', 'whisper', 'tts', 'embedding', 'clip',
    'lazy_loading',
]

caps = flux_model['metadata'].get('capabilities', [])
for cap in new_caps:
    if cap not in caps:
        caps.append(cap)
flux_model['metadata']['capabilities'] = caps

# Model stats
total_model_params = sum(
    m.get('total_params', 0) 
    for m in flux_model.get('models', {}).values()
)
flux_model['metadata']['total_model_params'] = total_model_params

print(f"  Version: {flux_model['version']}")
print(f"  Total model params: {total_model_params:,}")
print(f"  Capabilities: {len(caps)}")

log.cell_end("Cell 14 — Metadata", "PASS")

In [ ]:
"""Cell 15: Size Summary"""

log.cell_start("Cell 15 — Size Summary")

print("\n═══ EMBEDDED MODELS SUMMARY ═══\n")

total_size = 0
for name, state in flux_model.get('models', {}).items():
    params = state.get('total_params', 0)
    size_gb = params * 2 / 1e9  # fp16
    total_size += size_gb
    lazy = 'lazy' if state.get('lazy_load', True) else 'always'
    print(f"  {name:12}: {params:>12,} params ({size_gb:>5.2f} GB) [{lazy}]")

# Native components
print("\n═══ NATIVE COMPONENTS ═══\n")
native_size = 0
for key in ['cse', 'field', 'memory', 'bridges', 'decoder', 'causal', 'adapters', 'gravity']:
    if key in flux_model and flux_model[key]:
        size = get_component_size(flux_model[key])
        native_size += size
        print(f"  {key:12}: {size/1e6:>8.1f} MB")

print(f"\n═══ TOTALS ═══")
print(f"  Models: {total_size:.2f} GB")
print(f"  Native: {native_size/1e9:.2f} GB")
print(f"  Total:  {total_size + native_size/1e9:.2f} GB")

log.cell_end("Cell 15 — Size Summary", "PASS")

In [ ]:
"""Cell 16: Save Complete Model"""

log.cell_start("Cell 16 — Save")

OUTPUT_PATH = ROOT / 'checkpoints' / 'Flux-Apex-V1.flx'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Saving to: {OUTPUT_PATH}")
print("  This may take several minutes...")

torch.save(flux_model, str(OUTPUT_PATH), pickle_protocol=4)

size_gb = OUTPUT_PATH.stat().st_size / 1e9
print(f"\n  ✓ Saved: {OUTPUT_PATH.name}")
print(f"  Size: {size_gb:.2f} GB")

log.success(f"Model saved: {size_gb:.2f} GB")
log.cell_end("Cell 16 — Save", "PASS")

In [ ]:
"""Cell 17: Upload to HuggingFace (Optional)"""

log.cell_start("Cell 17 — Upload")

if hf_token:
    from huggingface_hub import HfApi
    
    api = HfApi(token=hf_token)
    
    print("Uploading to HuggingFace Hub...")
    print("  This may take 10-30 minutes for large files...")
    
    try:
        api.upload_file(
            path_or_fileobj=str(OUTPUT_PATH),
            path_in_repo='checkpoints/Flux-Apex-V1.flx',
            repo_id='UnseenGAP/FLUX',
            commit_message=f'Update Flux-Apex-V1.flx - v{flux_model["version"]} with all embedded models',
        )
        log.success("✓ Uploaded to HuggingFace Hub")
    except Exception as e:
        log.error(f"Upload failed: {e}")
else:
    print("  ⚠ No HF token - skipping upload")
    print("  Upload manually later:")
    print("    from flux_utils import upload_flx_to_hf")
    print("    upload_flx_to_hf('checkpoints/Flux-Apex-V1.flx')")

log.cell_end("Cell 17 — Upload", "PASS")

In [ ]:
"""Cell 18: Final Summary"""

log.separator("PHASE 2 COMPLETE")

print(f"""
═══════════════════════════════════════════════════════════════════════════
                    FLUX MODEL EMBEDDING - PHASE 2 COMPLETE
═══════════════════════════════════════════════════════════════════════════

  Output: {OUTPUT_PATH}
  Size:   {size_gb:.2f} GB
  Version: {flux_model['version']}

  Embedded Models:
  ┌─────────────┬─────────────────────────────────────┬──────────┬─────────┐
  │ Name        │ Base Model                          │ Params   │ Policy  │
  ├─────────────┼─────────────────────────────────────┼──────────┼─────────┤""")

for name, state in flux_model.get('models', {}).items():
    base = state.get('base_model', 'N/A')[:35]
    params = state.get('total_params', 0)
    params_str = f"{params/1e9:.1f}B" if params > 1e9 else f"{params/1e6:.0f}M"
    policy = 'always' if not state.get('lazy_load', True) else 'lazy'
    print(f"  │ {name:11} │ {base:35} │ {params_str:>8} │ {policy:7} │")

print(f"""  └─────────────┴─────────────────────────────────────┴──────────┴─────────┘

  VRAM Usage Scenarios:
  • Startup (instruct + embedding + clip):  ~4 GB
  • + Vision task:                          ~10 GB  
  • + Code task:                            ~13 GB
  • All models loaded:                      ~{total_size:.0f} GB

  Next Steps:
  1. Phase 2.5: Embed detection models (InsightFace, DINO, MiDaS, ViTPose)
  2. Phase 3: Run flux_lite_full_test.ipynb for validation
  3. Test offline operation (no network during inference)

═══════════════════════════════════════════════════════════════════════════
""")